# Logistic Regresssion

In [1]:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('https://raw.githubusercontent.com/marvv0905/ISOM3360-project/refs/heads/main/datasets/shopping_dataset_dummyCoded.csv')

### Split df into features and target variable

In [2]:
#encode Y, since multiclass value
encoder = LabelEncoder()

X = df.drop("shopping_preference", axis=1)
Y = encoder.fit_transform(df["shopping_preference"])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=42)

### Normalizing the numerical features

In [5]:
#exclude the binary features
dummy_cols = ['gender_Male', 'gender_Female', 'gender_Other','city_tier_Tier 1', 'city_tier_Tier 2', 'city_tier_Tier 3']
numeric_cols = [col for col in X.columns if col not in dummy_cols]

zscore_scaler = preprocessing.StandardScaler().fit(X_train[numeric_cols])
X_train[numeric_cols] = zscore_scaler.transform(X_train[numeric_cols])
X_test[numeric_cols] = zscore_scaler.transform(X_test[numeric_cols])


X_train.sample(5)

,age,monthly_income,daily_internet_hours,smartphone_usage_years,social_media_hours,online_payment_trust_score,tech_savvy_score,monthly_online_orders,monthly_store_visits,avg_online_spend,...,need_touch_feel_score,brand_loyalty_score,environmental_awareness,time_pressure_level,gender_Female,gender_Male,gender_Other,city_tier_Tier 1,city_tier_Tier 2,city_tier_Tier 3
4202,0.071460,-0.602032,1.614414,-0.385974,-1.586702,-0.518464,0.852972,-1.638891,-0.254553,-0.525457,...,0.859557,-0.542480,0.188891,-0.875941,False,True,False,True,False,False
6492,1.416118,1.208870,-1.218955,0.111380,-1.191547,-0.518464,1.542098,0.088187,-0.254553,-1.379850,...,0.511722,0.511238,-0.506660,0.167049,False,True,False,True,False,False
10434,1.528172,-1.569167,-0.156441,0.360057,1.100353,-0.866045,1.197535,1.331683,-1.131547,-1.335910,...,0.163886,-0.893720,-1.202212,-0.875941,False,True,False,False,True,False
9581,0.239542,0.265001,-0.965975,0.360057,-1.191547,1.567018,0.508410,0.364520,1.324035,1.056873,...,-1.575293,0.862478,-0.506660,-0.180615,False,False,True,True,False,False
652,0.407624,-0.607852,0.653093,0.857411,-0.164144,-1.213625,-0.180716,1.331683,0.797839,-1.224527,...,-0.879622,-0.191241,0.884443,-1.223604,False,True,False,True,False,False


## Base Model

In [ ]:
logistic_model_base = LogisticRegression().fit(X_train, y_train)

Train accuracy: 0.9829
Test accuracy:  0.9785
Gap:0.0044


In [22]:
train_performance = logistic_model_base.predict(X_train)
print("Training set performance")
print(classification_report(y_train, train_performance, target_names=encoder.classes_))
print(confusion_matrix(y_train, train_performance))

print("===========================================================")

test_performance  = logistic_model_base.predict(X_test)
print("Test set performance")
print(classification_report(y_test, test_performance, target_names=encoder.classes_))
print(confusion_matrix(y_test, test_performance))

print("===========================================================")

train_score = logistic_model_base.score(X_train, y_train)
test_score = logistic_model_base.score(X_test, y_test)

print(f"Train accuracy: {train_score:.4f}")
print(f"Test accuracy:  {test_score:.4f}")

Training set performance
              precision    recall  f1-score   support

      Hybrid       1.00      0.45      0.62       258
      Online       0.93      1.00      0.96       823
       Store       0.99      1.00      0.99      7171

    accuracy                           0.98      8252
   macro avg       0.97      0.82      0.86      8252
weighted avg       0.98      0.98      0.98      8252

[[ 117   60   81]
 [   0  823    0]
 [   0    0 7171]]
Test set performance
              precision    recall  f1-score   support

      Hybrid       0.93      0.34      0.50       111
      Online       0.94      0.99      0.96       353
       Store       0.98      1.00      0.99      3073

    accuracy                           0.98      3537
   macro avg       0.95      0.78      0.82      3537
weighted avg       0.98      0.98      0.97      3537

[[  38   24   49]
 [   3  350    0]
 [   0    0 3073]]
Train accuracy: 0.9829
Test accuracy:  0.9785


## Model Tuning 

### 1. Lasso regularlization

In [24]:
logistic_model_l1 = LogisticRegression(penalty='l1', C=1.0, max_iter=10000,
                                    random_state=0,
                                    solver='saga', 
                                    class_weight='balanced', 
                                    tol=0.01)

logistic_model_l1.fit(X_train, y_train)
logistic_l1_train_perform = logistic_model_l1.predict(X_train)
logistic_l1_test_perform = logistic_model_l1.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l1_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l1_test_perform, target_names=encoder.classes_))

print("===========================================================")

logistic_score_l1 = cross_val_score(logistic_model_l1, X_train, y_train, cv=10, scoring='f1_macro')
print(f"CV Macro F1: {logistic_score_l1.mean():.4f} ± {logistic_score_l1.std():.4f}")

Training set performance
              precision    recall  f1-score   support

      Hybrid       0.66      0.88      0.75       258
      Online       0.98      0.97      0.98       823
       Store       1.00      0.99      0.99      7171

    accuracy                           0.98      8252
   macro avg       0.88      0.95      0.91      8252
weighted avg       0.99      0.98      0.98      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.65      0.82      0.72       111
      Online       0.99      0.97      0.98       353
       Store       1.00      0.99      0.99      3073

    accuracy                           0.98      3537
   macro avg       0.88      0.93      0.90      3537
weighted avg       0.98      0.98      0.98      3537

CV Macro F1: 0.8527 ± 0.0210


#### Finding best 'c' for L1 model with GridsearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_logisticReg_l1 = GridSearchCV(estimator=LogisticRegression(penalty='l1', solver='saga',
                                  max_iter=10000, tol=0.01,
                                  class_weight='balanced',
                                  random_state=0),
                                  param_grid=param_grid,
                                  cv=5,
                                  scoring='f1_macro'
)

grid_logisticReg_l1.fit(X_train, y_train)

print("Best C:", grid_logisticReg_l1.best_params_)
print("Best CV Macro F1:", grid_logisticReg_l1.best_score_)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max

Best C: {'C': 10}
Best CV Macro F1: 0.8673740899033376


#### Applying the best 'c' for L1 model

In [27]:
logistic_model_l1 = LogisticRegression(penalty='l1', C=10, max_iter=10000,
                                    random_state=0,
                                    solver='saga', 
                                    class_weight='balanced', 
                                    tol=0.01)

logistic_model_l1.fit(X_train, y_train)
logistic_l1_train_perform = logistic_model_l1.predict(X_train)
logistic_l1_test_perform = logistic_model_l1.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l1_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l1_test_perform, target_names=encoder.classes_))

print(f"Train accuracy: {logistic_model_l1.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {logistic_model_l1.score(X_test, y_test):.4f}")


Training set performance
              precision    recall  f1-score   support

      Hybrid       0.51      0.96      0.67       258
      Online       0.99      0.94      0.97       823
       Store       1.00      0.97      0.99      7171

    accuracy                           0.97      8252
   macro avg       0.83      0.96      0.87      8252
weighted avg       0.98      0.97      0.97      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.53      0.99      0.69       111
      Online       1.00      0.93      0.96       353
       Store       1.00      0.98      0.99      3073

    accuracy                           0.97      3537
   macro avg       0.84      0.97      0.88      3537
weighted avg       0.98      0.97      0.98      3537

Train accuracy: 0.9701
Test accuracy:  0.9717


### 2. Ridge regularlization

In [26]:
logistic_model_l2 = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                                    max_iter=1000,
                                    class_weight='balanced',
                                    random_state=0)


logistic_model_l2.fit(X_train, y_train)
logistic_l2_train_perform = logistic_model_l2.predict(X_train)
logistic_l2_test_perform = logistic_model_l2.predict(X_test)

print("Training set performance")
print(classification_report(y_train, logistic_l2_train_perform, target_names=encoder.classes_))

print("===========================================================")

print("Testing set performance")
print(classification_report(y_test, logistic_l2_test_perform, target_names=encoder.classes_))

print("===========================================================")

logistic_score_l2 = cross_val_score(logistic_model_l2, X_train, y_train, cv=10, scoring='f1_macro')
print(f"CV Macro F1: {logistic_score_l2.mean():.4f} ± {logistic_score_l2.std():.4f}")

Training set performance
              precision    recall  f1-score   support

      Hybrid       0.40      1.00      0.57       258
      Online       1.00      0.94      0.97       823
       Store       1.00      0.95      0.98      7171

    accuracy                           0.95      8252
   macro avg       0.80      0.96      0.84      8252
weighted avg       0.98      0.95      0.96      8252

Testing set performance
              precision    recall  f1-score   support

      Hybrid       0.43      1.00      0.60       111
      Online       1.00      0.94      0.97       353
       Store       1.00      0.96      0.98      3073

    accuracy                           0.96      3537
   macro avg       0.81      0.97      0.85      3537
weighted avg       0.98      0.96      0.97      3537

CV Macro F1: 0.8315 ± 0.0156


In [ ]:
grid_logisticReg_l2 = GridSearchCV(estimator=LogisticRegression(penalty='l2', solver='lbfgs',
                                  max_iter=1000,
                                  class_weight='balanced',
                                  random_state=0),
                                  param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100]},
                                  cv=5,
                                  scoring='f1_macro'
)

grid_logisticReg_l2.fit(X_train, y_train)


print("Best C:", grid_logisticReg_l2.best_params_)
print("Best CV Macro F1:", grid_logisticReg_l2.best_score_)